# 05 — ML Feature Contract, Tuning, Model Lock, and Final ML Predictions

This notebook builds the TraceAdvisor ML layer on top of the canonical Notebook 04 occurrence artifacts.

## Pipeline position

- **N04** — corrected occurrence artifacts and baseline recommendations.
- **N05 (this notebook)** — tune and select the final ML models, refit the locked models, generate final ML predictions, and provide the ML/P95 inputs for authoritative evaluation.
- **N06** — evaluate Request/Mean/P90/P95/P99, receive the ML predictions, construct Hybrid recommendations, and compare all policies through the shared evaluator.

## Responsibilities

1. Load the canonical N04 train/validation occurrence artifacts and `metadata.json`; do not rebuild splits or targets.
2. Audit an explicit **feature manifest** containing only information available at the start of the current occurrence.
3. Build CPU and memory training rows **independently**, keeping only labels that belong to the training period.
4. Reproduce the N04 baseline validation metrics exactly as an integrity check.
5. Tune Decision Tree, Random Forest, and XGBoost regressors with label-purged `TimeSeriesSplit`; report `GroupKFold` separately as a robustness check.
6. Select one final model per resource on validation with a pre-declared rule.
7. Refit the locked models on train + validation and generate final ML predictions for the test occurrences.
8. Save the locked models, predictions, feature manifest, and metadata for N06.

### Final-test discipline

N05 never scores the test split. Test target and outcome columns are dropped immediately after loading, so no test metric can be computed here. N05 does not construct Hybrid recommendations or compare recommendation policies; that is N06's responsibility.

### Canonical modeling unit

One row is one **occurrence** (`occurrence_key = recurrence_key || execution_start_time_us`), exactly as in N04. Features describe the state at the start of the current occurrence; the target is the full-workload peak of the next distinct occurrence of the same `recurrence_key`.

In [1]:
from pathlib import Path
import json
import platform
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

In [2]:
# Locate repository root.
REPO_ROOT = Path.cwd()

while (
    REPO_ROOT.name != "masters-project"
    and REPO_ROOT.parent != REPO_ROOT
):
    REPO_ROOT = REPO_ROOT.parent

if REPO_ROOT.name != "masters-project":
    raise RuntimeError(
        "Could not locate masters-project repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)

Repository root: /Users/shivani/Documents/masters-project


In [3]:
from src.evaluation.evaluator import RecommendationEvaluator
from src.features.feature_config import (
    OCCURRENCE_FEATURE_COLS,
    OCCURRENCE_LEAKY_COLUMNS,
    OCCURRENCE_LEAKY_PREFIXES,
    OCCURRENCE_TARGET_CPU,
    OCCURRENCE_TARGET_MEMORY,
    OCCURRENCE_TARGET_TIME,
)
from src.features.ml_prep import (
    audit_occurrence_feature_manifest,
    build_occurrence_feature_matrix,
    purged_time_series_folds,
    select_label_period_rows,
)

DATA_DIR = REPO_ROOT / "data" / "processed" / "google_subset_v1"

# Notebook 04 canonical inputs.
TRAIN_OCCURRENCES_PATH = DATA_DIR / "train_occurrences.parquet"
VAL_OCCURRENCES_PATH = DATA_DIR / "val_occurrences.parquet"
TEST_OCCURRENCES_PATH = DATA_DIR / "test_occurrences.parquet"
N04_METADATA_PATH = DATA_DIR / "metadata.json"
N04_VALIDATION_COMPARISON_PATH = DATA_DIR / "baseline_validation_comparison.csv"

# Notebook 05 outputs.
FEATURE_MANIFEST_PATH = DATA_DIR / "ml_feature_manifest.json"
TUNING_RESULTS_PATH = DATA_DIR / "ml_tuning_results.csv"
CV_SUMMARY_PATH = DATA_DIR / "ml_cv_summary.csv"
MODEL_COMPARISON_PATH = DATA_DIR / "ml_validation_model_comparison.csv"
VAL_PREDICTIONS_PATH = DATA_DIR / "val_ml_predictions.parquet"
TEST_PREDICTIONS_PATH = DATA_DIR / "test_ml_predictions.parquet"
N05_METADATA_PATH = DATA_DIR / "ml_metadata.json"
LOCKED_MODEL_DIR = DATA_DIR / "n05_locked_models"

RANDOM_STATE = 42
N_CV_SPLITS = 5

print("Data directory:", DATA_DIR)
print("Feature count in manifest:", len(OCCURRENCE_FEATURE_COLS))

Data directory: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1
Feature count in manifest: 24


## Canonical N04 → N05 Contract

N05 consumes the N04 occurrence artifacts and metadata as-is:

- `train_occurrences.parquet` and `val_occurrences.parquet`: one row per occurrence, with point-in-time history, next-occurrence targets, and baseline recommendations at the N04-selected `min_runs`;
- `test_occurrences.parquet`: loaded only after the models are locked, and only for features, identifiers, and the P95 baseline;
- `metadata.json`: authoritative target field names, split boundaries, selected baseline, and the common validation population size;
- `baseline_validation_comparison.csv`: N04 validation metrics, used to prove that N05 reads the same artifacts N04 produced.

The target column names are read from `metadata.json` and must agree with the shared feature configuration.

In [4]:
for required_path in [
    TRAIN_OCCURRENCES_PATH,
    VAL_OCCURRENCES_PATH,
    TEST_OCCURRENCES_PATH,
    N04_METADATA_PATH,
    N04_VALIDATION_COMPARISON_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Notebook 04 output was not found. "
            f"Expected: {required_path}. "
            "Run 04_splitAndBaselines.ipynb first."
        )

with open(N04_METADATA_PATH, "r", encoding="utf-8") as file:
    n04_metadata = json.load(file)

train_occurrences = pd.read_parquet(TRAIN_OCCURRENCES_PATH)
val_occurrences = pd.read_parquet(VAL_OCCURRENCES_PATH)
n04_validation_comparison = pd.read_csv(N04_VALIDATION_COMPARISON_PATH)

TARGET_CPU_COL = n04_metadata["target_cpu_field"]
TARGET_MEMORY_COL = n04_metadata["target_memory_field"]
TARGET_TIME_COL = n04_metadata["target_timestamp_field"]

if (TARGET_CPU_COL, TARGET_MEMORY_COL, TARGET_TIME_COL) != (
    OCCURRENCE_TARGET_CPU,
    OCCURRENCE_TARGET_MEMORY,
    OCCURRENCE_TARGET_TIME,
):
    raise RuntimeError(
        "N04 metadata target fields disagree with src/features/feature_config.py."
    )

# N04 scores every policy against these same next-occurrence targets.
assert n04_metadata["evaluation_actual_cpu_field"] == TARGET_CPU_COL
assert n04_metadata["evaluation_actual_memory_field"] == TARGET_MEMORY_COL
assert n04_metadata["canonical_modeling_unit"] == "recurrence_occurrence"
assert n04_metadata["baseline_selection_scope"] == "validation_only"

SELECTED_MIN_RUNS = int(n04_metadata["selected_min_runs"])
SELECTED_BASELINE_POLICY = str(n04_metadata["selected_baseline_policy"])

split_boundaries = n04_metadata["split_boundaries"]
TRAIN_END_TIME = int(split_boundaries["train_end"])
VAL_START_TIME = int(split_boundaries["val_start"])
VAL_END_TIME = int(split_boundaries["val_end"])
TEST_START_TIME = int(split_boundaries["test_start"])

RESOURCE_TARGETS = {
    "cpu": TARGET_CPU_COL,
    "memory": TARGET_MEMORY_COL,
}

print("Train occurrences:", train_occurrences.shape)
print("Validation occurrences:", val_occurrences.shape)
print("Target CPU field:", TARGET_CPU_COL)
print("Target memory field:", TARGET_MEMORY_COL)
print("N04 selected baseline:", SELECTED_BASELINE_POLICY, "| min_runs:", SELECTED_MIN_RUNS)
print("Train end:", TRAIN_END_TIME, "| Validation:", VAL_START_TIME, "->", VAL_END_TIME)
print("Test start:", TEST_START_TIME)

Train occurrences: (3473, 73)
Validation occurrences: (675, 73)
Target CPU field: target_cpu_next_occurrence_peak_ncu
Target memory field: target_memory_next_occurrence_peak_normalized
N04 selected baseline: P99 | min_runs: 1
Train end: 1812600000000 | Validation: 1813200000000 -> 2261100000000
Test start: 2262600000000


In [5]:
REQUIRED_OCCURRENCE_COLUMNS = (
    [
        "occurrence_key",
        "occurrence_id",
        "recurrence_key",
        "execution_start_time_us",
        TARGET_CPU_COL,
        TARGET_MEMORY_COL,
        TARGET_TIME_COL,
        "rec_cpu_request",
        "rec_cpu_mean",
        "rec_cpu_p90",
        "rec_cpu_p95",
        "rec_cpu_p99",
        "rec_memory_request",
        "rec_memory_mean",
        "rec_memory_p90",
        "rec_memory_p95",
        "rec_memory_p99",
    ]
    + OCCURRENCE_FEATURE_COLS
)


def validate_occurrence_schema(df: pd.DataFrame, split_name: str) -> None:
    missing_columns = [
        column for column in REQUIRED_OCCURRENCE_COLUMNS
        if column not in df.columns
    ]

    # Missing features fail loudly; they are never silently dropped.
    if missing_columns:
        raise ValueError(
            f"[{split_name}] Missing required occurrence columns: {missing_columns}"
        )

    if not df["occurrence_key"].is_unique:
        raise ValueError(f"[{split_name}] occurrence_key must be unique.")

    if not (
        df["occurrence_key"].astype(str) == df["occurrence_id"].astype(str)
    ).all():
        raise ValueError(f"[{split_name}] occurrence_id must equal occurrence_key.")

    target_available = df[TARGET_TIME_COL].notna()

    # Every target must point to a strictly later occurrence.
    if not (
        df.loc[target_available, TARGET_TIME_COL]
        > df.loc[target_available, "execution_start_time_us"]
    ).all():
        raise ValueError(f"[{split_name}] A target does not point forward in time.")

    print(f"[{split_name}] Occurrence schema validation passed.")


validate_occurrence_schema(train_occurrences, "train")
validate_occurrence_schema(val_occurrences, "validation")

# Occurrences must stay inside their frozen N04 split boundaries.
assert (train_occurrences["execution_start_time_us"] <= TRAIN_END_TIME).all()
assert val_occurrences["execution_start_time_us"].between(
    VAL_START_TIME,
    VAL_END_TIME,
).all()
assert set(train_occurrences["occurrence_key"]).isdisjoint(
    set(val_occurrences["occurrence_key"])
)

print("Split boundary and disjointness checks passed.")

[train] Occurrence schema validation passed.
[validation] Occurrence schema validation passed.
Split boundary and disjointness checks passed.


## Feature Manifest and Leakage Audit

The feature manifest is defined once in `src/features/feature_config.py` (`OCCURRENCE_FEATURE_COLS`), and the audit and matrix construction live in `src/features/ml_prep.py`, so N06 and later inference use the same inputs in the same order.

Every feature is known at the start of the current occurrence:

- the current occurrence request, number of parallel executions, and request heterogeneity;
- Notebook 03 completed-occurrence history for CPU, memory, and runtime.

The following are never model inputs: the current occurrence's own peak, end time, and runtime (unknown at its start), every `target_*` column, every `rec_*` baseline recommendation, and every `uses_history_*` flag.

Missing values stay missing. Decision Tree, Random Forest, and XGBoost handle `NaN` natively, so nothing is imputed and no learned preprocessing exists that could be fit outside the training rows.

In [6]:
FEATURE_COLS = list(OCCURRENCE_FEATURE_COLS)

feature_audit = audit_occurrence_feature_manifest(
    {
        "train": train_occurrences,
        "validation": val_occurrences,
    },
    target_cols=[TARGET_CPU_COL, TARGET_MEMORY_COL],
    feature_cols=FEATURE_COLS,
)

print("Leakage audit passed:", len(FEATURE_COLS), "features.")
print("No target, current-outcome, rec_*, or uses_history_* column is a feature.")

display(
    feature_audit.pivot(
        index="feature",
        columns="split",
        values="missing_rows",
    ).loc[FEATURE_COLS]
)

Leakage audit passed: 24 features.
No target, current-outcome, rec_*, or uses_history_* column is a feature.


split,train,validation
feature,,
occurrence_requested_cpu_ncu,50,14
occurrence_requested_memory_normalized,50,14
occurrence_execution_count,0,0
occurrence_cpu_request_nunique,0,0
occurrence_memory_request_nunique,0,0
prior_completed_occurrence_count,0,0
occ_hist_peak_cpu_count,0,0
occ_hist_peak_cpu_mean,1801,239
occ_hist_peak_cpu_std,2283,324


## Training Rows: Labels Must Belong to the Training Period

N04 builds next-occurrence targets before splitting, so the last training occurrence of a recurrence can have a target occurrence that starts in the validation period or later.

A model trained on that label would learn from a validation-period outcome before validation is scored. Therefore the **primary** training rows keep only labels whose target occurrence starts at or before the training boundary (`target_next_occurrence_start_time_us <= train_end`).

CPU and memory rows are built independently: a row with a valid CPU target remains usable for CPU even when its memory target is missing.

A **sensitivity** variant that keeps every available training label is reported later. It is never used for model selection.

In [7]:
PRIMARY_LABEL_RULE = "train_period_only"

training_rows = {}

for resource, target_col in RESOURCE_TARGETS.items():
    training_rows[(resource, "train_period_only")] = select_label_period_rows(
        train_occurrences,
        target_col=target_col,
        target_time_col=TARGET_TIME_COL,
        label_cutoff_time=TRAIN_END_TIME,
    )
    training_rows[(resource, "all_available")] = select_label_period_rows(
        train_occurrences,
        target_col=target_col,
        target_time_col=TARGET_TIME_COL,
        label_cutoff_time=None,
    )

for (resource, rule), rows in training_rows.items():
    target_col = RESOURCE_TARGETS[resource]

    assert rows["occurrence_key"].is_unique
    assert rows[target_col].notna().all()
    assert (rows[target_col] >= 0).all()

    if rule == "train_period_only":
        assert (rows[TARGET_TIME_COL] <= TRAIN_END_TIME).all()

training_row_summary = pd.DataFrame(
    [
        {
            "resource": resource,
            "label_period_rule": rule,
            "training_rows": len(rows),
            "cold_start_rows": int(
                (rows["prior_completed_occurrence_count"] == 0).sum()
            ),
            "recurrence_keys": rows["recurrence_key"].nunique(),
        }
        for (resource, rule), rows in training_rows.items()
    ]
)

display(training_row_summary)

excluded_cpu = (
    len(training_rows[("cpu", "all_available")])
    - len(training_rows[("cpu", "train_period_only")])
)
print("CPU labels excluded because the target lies after the training period:", excluded_cpu)

,resource,label_period_rule,training_rows,cold_start_rows,recurrence_keys
0,cpu,train_period_only,1672,482,482
1,cpu,all_available,2010,618,618
2,memory,train_period_only,1649,475,477
3,memory,all_available,1983,611,615


CPU labels excluded because the target lies after the training period: 338


## Common Validation Population

The validation population is rebuilt with the exact N04 rule: validation occurrences whose next-occurrence CPU **and** memory targets are both available. Model selection uses these same occurrence IDs, and N06 scores policies on them.

In [8]:
COMMON_EVAL_REQUIRED = n04_metadata["common_validation_population"]["required_fields"]
assert COMMON_EVAL_REQUIRED == [TARGET_CPU_COL, TARGET_MEMORY_COL]

validation_common_mask = (
    val_occurrences[COMMON_EVAL_REQUIRED]
    .notna()
    .all(axis=1)
)

val_eval = (
    val_occurrences.loc[validation_common_mask]
    .sort_values(["execution_start_time_us", "occurrence_key"], kind="mergesort")
    .reset_index(drop=True)
)

expected_common_count = int(
    n04_metadata["common_validation_population"]["occurrences"]
)

if len(val_eval) != expected_common_count:
    raise RuntimeError(
        f"Common validation population has {len(val_eval)} occurrences; "
        f"N04 recorded {expected_common_count}."
    )

VALIDATION_EVAL_IDS = set(val_eval["occurrence_id"])

print("Validation occurrences:", len(val_occurrences))
print("Common evaluated occurrences:", len(val_eval))
print(
    "Common occurrences with no prior history:",
    int((val_eval["prior_completed_occurrence_count"] == 0).sum()),
)

Validation occurrences: 675
Common evaluated occurrences: 329
Common occurrences with no prior history: 26


## Integrity Check: Reproduce the N04 Baseline Validation Metrics

The N04 baselines are re-scored with the shared evaluator on the common population. Every metric must equal the N04 `baseline_validation_comparison.csv` row for the selected `min_runs`. This proves N05 reads exactly the artifacts N04 produced; the authoritative policy comparison itself belongs to N06.

In [9]:
evaluator = RecommendationEvaluator()

BASELINE_POLICIES = {
    "Request": ("rec_cpu_request", "rec_memory_request"),
    "Mean": ("rec_cpu_mean", "rec_memory_mean"),
    "P90": ("rec_cpu_p90", "rec_memory_p90"),
    "P95": ("rec_cpu_p95", "rec_memory_p95"),
    "P99": ("rec_cpu_p99", "rec_memory_p99"),
}

n04_selected_rows = n04_validation_comparison[
    n04_validation_comparison["min_runs"] == SELECTED_MIN_RUNS
].set_index("policy")

for policy_name, (cpu_col, memory_col) in BASELINE_POLICIES.items():
    evaluated = evaluator.evaluate(
        data=val_eval,
        actual_cpu_col=TARGET_CPU_COL,
        actual_memory_col=TARGET_MEMORY_COL,
        recommended_cpu_col=cpu_col,
        recommended_memory_col=memory_col,
        policy_name=policy_name,
        id_cols=["occurrence_id", "recurrence_key"],
    )
    summary = evaluator.summarize(evaluated)
    expected = n04_selected_rows.loc[policy_name]

    for metric, value in summary.items():
        if not np.isclose(
            float(value),
            float(expected[metric]),
            rtol=1e-9,
            atol=1e-12,
            equal_nan=True,
        ):
            raise RuntimeError(
                f"{policy_name} {metric}: N05={value} but N04={expected[metric]}."
            )

print(
    "All N04 baseline validation metrics reproduced exactly "
    f"(min_runs={SELECTED_MIN_RUNS}, {len(BASELINE_POLICIES)} policies)."
)

All N04 baseline validation metrics reproduced exactly (min_runs=1, 5 policies).


## Candidate Models, Target Transform, and Selection Rule

Three regressor families are tuned separately for CPU and memory: Decision Tree, Random Forest, and XGBoost. The fixed starting hyperparameters from the earlier subset ML notebook are also scored on validation for reference, together with a median reference predictor. Neither is selectable.

### Target transform

Next-occurrence peaks are heavily right-skewed: most occurrences are tiny, while a few distributed occurrences are orders of magnitude larger. Models are trained on `log1p(target)`, and predictions are mapped back with `expm1` and clipped at zero. The transform is fixed, not learned.

### Pre-declared rules

1. **Hyperparameters** per family and resource: lowest mean RMSE on the `log1p` scale across label-purged `TimeSeriesSplit` folds of the training rows.
2. **Final model** per resource: among the tuned families, lowest validation RMSE on the `log1p` scale over the common validation population, with lower raw-scale MAE as the tie-breaker.
3. `GroupKFold` grouped by `recurrence_key` is reported separately as a robustness check of generalization to unseen recurrences. It is not used for any decision, because its folds are not chronological.

Raw-scale MAE, RMSE, and R² are also reported, because raw RMSE is dominated by a handful of very large occurrences.

In [10]:
CANDIDATE_MODELS = ["decision_tree", "random_forest", "xgboost"]
REFERENCE_MODEL = "reference_median"

# Fixed arguments shared by every fit. Models use one thread each so that
# GridSearchCV can parallelize across candidates deterministically.
FIXED_PARAMS = {
    "decision_tree": {"random_state": RANDOM_STATE},
    "random_forest": {"random_state": RANDOM_STATE, "n_jobs": 1},
    "xgboost": {
        "objective": "reg:squarederror",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
    },
}

# Starting hyperparameters from the earlier subset ML notebook (reference only).
STARTING_PARAMS = {
    "decision_tree": {"max_depth": 8, "min_samples_leaf": 10},
    "random_forest": {"n_estimators": 200, "max_depth": 12, "min_samples_leaf": 5},
    "xgboost": {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
}

PARAM_GRIDS = {
    "decision_tree": {
        "max_depth": [4, 6, 8, 12],
        "min_samples_leaf": [5, 10, 20, 40],
    },
    "random_forest": {
        "n_estimators": [300],
        "max_depth": [6, 10, None],
        "min_samples_leaf": [2, 5, 10],
        "max_features": [1.0, 0.5, "sqrt"],
    },
    "xgboost": {
        "n_estimators": [200, 500],
        "max_depth": [3, 5],
        "learning_rate": [0.03, 0.1],
        "subsample": [0.8],
        "colsample_bytree": [0.8, 1.0],
        "min_child_weight": [1, 5],
    },
}


def make_model(model_name: str, params: dict | None = None):
    params = {**FIXED_PARAMS.get(model_name, {}), **(params or {})}

    if model_name == "decision_tree":
        return DecisionTreeRegressor(**params)
    if model_name == "random_forest":
        return RandomForestRegressor(**params)
    if model_name == "xgboost":
        return XGBRegressor(**params)
    if model_name == REFERENCE_MODEL:
        return DummyRegressor(strategy="median")
    raise ValueError(f"Unknown model: {model_name}")


def log_target(df: pd.DataFrame, target_col: str) -> np.ndarray:
    return np.log1p(df[target_col].to_numpy(dtype=float))


def predict_resource(model, df: pd.DataFrame) -> np.ndarray:
    log_prediction = model.predict(build_occurrence_feature_matrix(df, FEATURE_COLS))
    return np.clip(np.expm1(log_prediction), 0.0, None)


for model_name, grid in PARAM_GRIDS.items():
    n_combinations = int(np.prod([len(values) for values in grid.values()]))
    print(f"{model_name}: {n_combinations} hyperparameter combinations")

decision_tree: 16 hyperparameter combinations
random_forest: 27 hyperparameter combinations
xgboost: 32 hyperparameter combinations


## Hyperparameter Tuning with Label-Purged TimeSeriesSplit

Folds come from `purged_time_series_folds` in `src/features/ml_prep.py`: an expanding-window `TimeSeriesSplit` over complete timestamp groups. In each fold, training rows whose target occurrence starts at or after the fold's validation block are purged, applying the same label-period rule used for the primary training rows.

Only training rows are used. The validation split is not touched during tuning.

In [11]:
tuning_results = []
tuned_searches = {}

for resource, target_col in RESOURCE_TARGETS.items():
    rows = training_rows[(resource, PRIMARY_LABEL_RULE)]
    X_train = build_occurrence_feature_matrix(rows, FEATURE_COLS)
    y_train = log_target(rows, target_col)

    folds = purged_time_series_folds(
        rows,
        target_time_col=TARGET_TIME_COL,
        n_splits=N_CV_SPLITS,
    )

    print(
        f"[{resource}] fold sizes (train/validation):",
        [(len(train_index), len(val_index)) for train_index, val_index in folds],
    )

    for model_name in CANDIDATE_MODELS:
        search = GridSearchCV(
            estimator=make_model(model_name),
            param_grid=PARAM_GRIDS[model_name],
            scoring="neg_root_mean_squared_error",
            cv=folds,
            n_jobs=-1,
            refit=True,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        tuned_searches[(resource, model_name)] = search

        cv_results = pd.DataFrame(search.cv_results_)

        for _, result in cv_results.iterrows():
            tuning_results.append(
                {
                    "resource": resource,
                    "model": model_name,
                    "params": json.dumps(result["params"], sort_keys=True),
                    "tss_mean_log_rmse": -result["mean_test_score"],
                    "tss_std_log_rmse": result["std_test_score"],
                    "rank": int(result["rank_test_score"]),
                }
            )

tuning_results = pd.DataFrame(tuning_results)

best_params = {
    key: search.best_params_
    for key, search in tuned_searches.items()
}

display(
    tuning_results[tuning_results["rank"] == 1]
    .sort_values(["resource", "tss_mean_log_rmse"])
    .reset_index(drop=True)
    .round(5)
)

[cpu] fold sizes (train/validation): [(154, 277), (350, 274), (596, 270), (850, 275), (1172, 270)]


[memory] fold sizes (train/validation): [(150, 274), (349, 272), (587, 267), (841, 272), (1158, 266)]


,resource,model,params,tss_mean_log_rmse,tss_std_log_rmse,rank
0,cpu,random_forest,"{""max_depth"": 10, ""max_features"": ""sqrt"", ""min...",0.83907,0.05437,1
1,cpu,decision_tree,"{""max_depth"": 4, ""min_samples_leaf"": 40}",0.85282,0.03247,1
2,cpu,xgboost,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",0.87486,0.05922,1
3,memory,random_forest,"{""max_depth"": 6, ""max_features"": ""sqrt"", ""min_...",0.47134,0.07721,1
4,memory,decision_tree,"{""max_depth"": 6, ""min_samples_leaf"": 40}",0.47538,0.05934,1
5,memory,xgboost,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",0.49213,0.08396,1


## Separate Cross-Validation Reports

`TimeSeriesSplit` and `GroupKFold` answer different questions and are reported side by side, never combined:

- **TimeSeriesSplit** (used for tuning): how well a model trained on earlier occurrences predicts later ones.
- **GroupKFold** (robustness only): how well the tuned model generalizes to recurrences it never saw in training.

In [12]:
cv_summary_rows = []

for (resource, model_name), search in tuned_searches.items():
    rows = training_rows[(resource, PRIMARY_LABEL_RULE)]
    X_train = build_occurrence_feature_matrix(rows, FEATURE_COLS)
    y_train = log_target(rows, RESOURCE_TARGETS[resource])

    group_scores = -cross_val_score(
        make_model(model_name, best_params[(resource, model_name)]),
        X_train,
        y_train,
        groups=rows["recurrence_key"],
        cv=GroupKFold(n_splits=N_CV_SPLITS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

    best_index = search.best_index_
    cv_summary_rows.append(
        {
            "resource": resource,
            "model": model_name,
            "best_params": json.dumps(search.best_params_, sort_keys=True),
            "tss_mean_log_rmse": -search.cv_results_["mean_test_score"][best_index],
            "tss_std_log_rmse": search.cv_results_["std_test_score"][best_index],
            "groupkfold_mean_log_rmse": group_scores.mean(),
            "groupkfold_std_log_rmse": group_scores.std(),
        }
    )

cv_summary = pd.DataFrame(cv_summary_rows)

display(cv_summary.round(5))

,resource,model,best_params,tss_mean_log_rmse,tss_std_log_rmse,groupkfold_mean_log_rmse,groupkfold_std_log_rmse
0,cpu,decision_tree,"{""max_depth"": 4, ""min_samples_leaf"": 40}",0.85282,0.03247,0.79402,0.27024
1,cpu,random_forest,"{""max_depth"": 10, ""max_features"": ""sqrt"", ""min...",0.83907,0.05437,0.78123,0.28465
2,cpu,xgboost,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",0.87486,0.05922,0.80670,0.31989
3,memory,decision_tree,"{""max_depth"": 6, ""min_samples_leaf"": 40}",0.47538,0.05934,0.44169,0.17448
4,memory,random_forest,"{""max_depth"": 6, ""max_features"": ""sqrt"", ""min_...",0.47134,0.07721,0.44154,0.18775
5,memory,xgboost,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",0.49213,0.08396,0.45138,0.20005


## Validation Comparison and Final Model Selection

Each tuned model (refit by `GridSearchCV` on all primary training rows) is scored on the common validation population. The starting hyperparameters and the median reference are shown for context only.

In [13]:
def prediction_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict:
    log_actual = np.log1p(actual)
    log_predicted = np.log1p(predicted)

    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
        "r2": r2_score(actual, predicted),
        "log_mae": mean_absolute_error(log_actual, log_predicted),
        "log_rmse": float(np.sqrt(mean_squared_error(log_actual, log_predicted))),
        "log_r2": r2_score(log_actual, log_predicted),
    }


model_comparison_rows = []
validation_predictions = {}

for resource, target_col in RESOURCE_TARGETS.items():
    actual = val_eval[target_col].to_numpy(dtype=float)
    primary_rows = training_rows[(resource, PRIMARY_LABEL_RULE)]

    variants = []

    for model_name in CANDIDATE_MODELS:
        variants.append(
            ("tuned", model_name, True, tuned_searches[(resource, model_name)].best_estimator_)
        )

        starting_model = make_model(model_name, STARTING_PARAMS[model_name])
        starting_model.fit(
            build_occurrence_feature_matrix(primary_rows, FEATURE_COLS),
            log_target(primary_rows, target_col),
        )
        variants.append(("starting", model_name, False, starting_model))

    reference_model = make_model(REFERENCE_MODEL)
    reference_model.fit(
        build_occurrence_feature_matrix(primary_rows, FEATURE_COLS),
        log_target(primary_rows, target_col),
    )
    variants.append(("reference", REFERENCE_MODEL, False, reference_model))

    for variant, model_name, selectable, model in variants:
        prediction = predict_resource(model, val_eval)

        assert len(prediction) == len(val_eval)
        assert np.isfinite(prediction).all()

        validation_predictions[(resource, variant, model_name)] = prediction

        model_comparison_rows.append(
            {
                "resource": resource,
                "variant": variant,
                "model": model_name,
                "selectable": selectable,
                "label_period_rule": PRIMARY_LABEL_RULE,
                "training_rows": len(primary_rows),
                "validation_rows": len(val_eval),
                **prediction_metrics(actual, prediction),
            }
        )

model_comparison = pd.DataFrame(model_comparison_rows)

display(
    model_comparison
    .sort_values(["resource", "log_rmse", "mae"], kind="mergesort")
    .reset_index(drop=True)
    .round(5)
)

,resource,variant,model,selectable,label_period_rule,training_rows,validation_rows,mae,rmse,r2,log_mae,log_rmse,log_r2
0,cpu,tuned,random_forest,True,train_period_only,1672,329,2.82362,9.55945,-0.00527,0.56623,0.85044,0.13701
1,cpu,tuned,xgboost,True,train_period_only,1672,329,2.85130,9.60001,-0.01381,0.56530,0.86044,0.11660
2,cpu,starting,random_forest,False,train_period_only,1672,329,2.87918,9.57507,-0.00855,0.57680,0.86570,0.10577
3,cpu,tuned,decision_tree,True,train_period_only,1672,329,2.85244,9.68647,-0.03216,0.56677,0.89099,0.05276
4,cpu,starting,xgboost,False,train_period_only,1672,329,2.96323,9.63688,-0.02162,0.59733,0.90545,0.02176
5,cpu,starting,decision_tree,False,train_period_only,1672,329,3.01729,9.80770,-0.05816,0.61423,0.98122,-0.14881
6,cpu,reference,reference_median,False,train_period_only,1672,329,2.75563,9.91415,-0.08125,0.54610,1.04771,-0.30977
7,memory,tuned,random_forest,True,train_period_only,1649,329,0.45380,1.25535,0.10741,0.22287,0.38703,0.18460
8,memory,starting,random_forest,False,train_period_only,1649,329,0.49564,1.27816,0.07467,0.24338,0.41533,0.06099
9,memory,tuned,decision_tree,True,train_period_only,1649,329,0.48776,1.31277,0.02388,0.23984,0.42139,0.03339


In [14]:
selection_pool = model_comparison[model_comparison["selectable"]]

SELECTED_MODELS = {}

for resource in RESOURCE_TARGETS:
    ranked = selection_pool[selection_pool["resource"] == resource].sort_values(
        ["log_rmse", "mae"],
        kind="mergesort",
    )
    SELECTED_MODELS[resource] = str(ranked.iloc[0]["model"])

LOCKED_PARAMS = {
    resource: {
        **FIXED_PARAMS[model_name],
        **best_params[(resource, model_name)],
    }
    for resource, model_name in SELECTED_MODELS.items()
}

print("Selected CPU model:", SELECTED_MODELS["cpu"], LOCKED_PARAMS["cpu"])
print("Selected memory model:", SELECTED_MODELS["memory"], LOCKED_PARAMS["memory"])

display(
    selection_pool[
        selection_pool.apply(
            lambda row: row["model"] == SELECTED_MODELS[row["resource"]],
            axis=1,
        )
    ].round(5)
)

Selected CPU model: random_forest {'random_state': 42, 'n_jobs': 1, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 300}
Selected memory model: random_forest {'random_state': 42, 'n_jobs': 1, 'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 300}


,resource,variant,model,selectable,label_period_rule,training_rows,validation_rows,mae,rmse,r2,log_mae,log_rmse,log_r2
2,cpu,tuned,random_forest,True,train_period_only,1672,329,2.82362,9.55945,-0.00527,0.56623,0.85044,0.13701
9,memory,tuned,random_forest,True,train_period_only,1649,329,0.45380,1.25535,0.10741,0.22287,0.38703,0.18460


## Sensitivity: Training on All Available Labels

The selected configurations are refit on every training label, including labels whose target occurrence lies in the validation period, and scored on the same validation rows. This shows how much the label-period rule changes validation accuracy. It is **not** used for any decision.

In [15]:
sensitivity_rows = []

for resource, model_name in SELECTED_MODELS.items():
    target_col = RESOURCE_TARGETS[resource]
    actual = val_eval[target_col].to_numpy(dtype=float)

    for rule in [PRIMARY_LABEL_RULE, "all_available"]:
        rows = training_rows[(resource, rule)]
        model = make_model(model_name, best_params[(resource, model_name)])
        model.fit(
            build_occurrence_feature_matrix(rows, FEATURE_COLS),
            log_target(rows, target_col),
        )

        sensitivity_rows.append(
            {
                "resource": resource,
                "model": model_name,
                "label_period_rule": rule,
                "training_rows": len(rows),
                **prediction_metrics(actual, predict_resource(model, val_eval)),
            }
        )

sensitivity = pd.DataFrame(sensitivity_rows)

display(sensitivity.round(5))

,resource,model,label_period_rule,training_rows,mae,rmse,r2,log_mae,log_rmse,log_r2
0,cpu,random_forest,train_period_only,1672,2.82362,9.55945,-0.00527,0.56623,0.85044,0.13701
1,cpu,random_forest,all_available,2010,2.83404,9.50694,0.00575,0.56932,0.84250,0.15306
2,memory,random_forest,train_period_only,1649,0.45380,1.25535,0.10741,0.22287,0.38703,0.18460
3,memory,random_forest,all_available,1983,0.45454,1.24966,0.11547,0.22377,0.38630,0.18766


## Model Findings Before Lock

These observations describe the run above. Recommendation-policy results (violations, waste, Hybrid) are produced in N06.

- **Random Forest is selected for both resources**, and it also has the lowest label-purged `TimeSeriesSplit` error during tuning (CPU 0.839, memory 0.471 `log1p` RMSE), so tuning and validation agree.
- **Tuning helps modestly.** Validation `log1p` RMSE improves from 0.866 to 0.850 for CPU and from 0.415 to 0.387 for memory relative to the starting hyperparameters; `log1p` R² rises to 0.14 (CPU) and 0.18 (memory).
- **Point accuracy remains weak on the raw scale.** Raw R² is about 0 for CPU and 0.11 for memory, and the constant median reference still has a lower raw MAE on both resources. The models rank and scale occurrences better than a constant, but they are not precise sizing rules on their own.
- **Generalization varies by workload.** `GroupKFold` means are close to the `TimeSeriesSplit` means, but their fold-to-fold spread is large (about 0.28 for CPU and 0.19 for memory), so accuracy differs strongly between recurrences.
- **The label-period rule changes little.** Training on all labels moves validation `log1p` RMSE from 0.850 to 0.843 (CPU) and from 0.387 to 0.386 (memory).
- **Caveat.** The final model was chosen on the same 329 validation occurrences it is scored on, so the validation numbers are optimistic. The locked models are evaluated once on the untouched test split in N06.

## Lock and Refit the Final Models on Train + Validation

The locked model class, hyperparameters, feature order, and target transform are now fixed. Each locked model is refit on training **and** validation rows.

The same label-period rule applies at the new boundary: a label is kept only when its target occurrence starts at or before the end of the validation period, so no refit label comes from the test period.

In [16]:
refit_source = pd.concat(
    [train_occurrences, val_occurrences],
    ignore_index=True,
)
assert refit_source["occurrence_key"].is_unique

refit_rows = {
    resource: select_label_period_rows(
        refit_source,
        target_col=target_col,
        target_time_col=TARGET_TIME_COL,
        label_cutoff_time=VAL_END_TIME,
    )
    for resource, target_col in RESOURCE_TARGETS.items()
}

locked_models = {}

for resource, model_name in SELECTED_MODELS.items():
    rows = refit_rows[resource]

    assert (rows[TARGET_TIME_COL] <= VAL_END_TIME).all()
    assert (rows["execution_start_time_us"] <= VAL_END_TIME).all()

    model = make_model(model_name, best_params[(resource, model_name)])
    model.fit(
        build_occurrence_feature_matrix(rows, FEATURE_COLS),
        log_target(rows, RESOURCE_TARGETS[resource]),
    )
    locked_models[resource] = model

display(
    pd.DataFrame(
        [
            {
                "resource": resource,
                "locked_model": SELECTED_MODELS[resource],
                "refit_rows": len(rows),
                "from_train_split": int(
                    (rows["execution_start_time_us"] <= TRAIN_END_TIME).sum()
                ),
                "from_validation_split": int(
                    (rows["execution_start_time_us"] >= VAL_START_TIME).sum()
                ),
            }
            for resource, rows in refit_rows.items()
        ]
    )
)

,resource,locked_model,refit_rows,from_train_split,from_validation_split
0,cpu,random_forest,2108,1882,226
1,memory,random_forest,2082,1857,225


## Final ML Predictions for the Test Occurrences

The test artifact is loaded only now, after every model decision is locked. Target and current-outcome columns are dropped immediately, so this notebook cannot compute any test metric. The locked models predict every test occurrence; N06 joins the targets and defines the scored population.

The P95 baseline (`rec_*_p95` at the N04-selected `min_runs`) is passed through unchanged as the Hybrid safeguard input.

In [17]:
TEST_COLUMNS_KEPT = [
    "occurrence_key",
    "occurrence_id",
    "recurrence_key",
    "execution_start_time_us",
    "rec_cpu_p95",
    "rec_memory_p95",
] + [column for column in FEATURE_COLS if column not in {"occurrence_key"}]

test_raw = pd.read_parquet(TEST_OCCURRENCES_PATH)
test_row_count = len(test_raw)

missing_test_columns = [
    column for column in TEST_COLUMNS_KEPT
    if column not in test_raw.columns
]
if missing_test_columns:
    raise ValueError(f"Test artifact is missing columns: {missing_test_columns}")

test_features = test_raw[list(dict.fromkeys(TEST_COLUMNS_KEPT))].copy()
del test_raw

leaked_test_columns = [
    column for column in test_features.columns
    if column in OCCURRENCE_LEAKY_COLUMNS
    or column.startswith(("target_",))
]
assert not leaked_test_columns, leaked_test_columns

assert len(test_features) == test_row_count
assert test_features["occurrence_key"].is_unique
assert (test_features["execution_start_time_us"] >= TEST_START_TIME).all()
assert set(test_features["occurrence_key"]).isdisjoint(
    set(refit_source["occurrence_key"])
)

audit_occurrence_feature_manifest(
    {"test": test_features},
    target_cols=[],
    feature_cols=FEATURE_COLS,
)

test_predictions = test_features[
    ["occurrence_key", "occurrence_id", "recurrence_key", "execution_start_time_us",
     "prior_completed_occurrence_count"]
].copy()

for resource, model in locked_models.items():
    prediction = predict_resource(model, test_features)

    assert len(prediction) == len(test_features)
    assert np.isfinite(prediction).all()

    test_predictions[f"ml_pred_{resource}"] = prediction

test_predictions["p95_cpu"] = test_features["rec_cpu_p95"].to_numpy()
test_predictions["p95_memory"] = test_features["rec_memory_p95"].to_numpy()

print("Test occurrences predicted:", len(test_predictions))
print("Test target columns available in this notebook:", [
    column for column in test_predictions.columns if column.startswith("target_")
])
print(
    "Missing P95 inputs (cpu/memory):",
    int(test_predictions["p95_cpu"].isna().sum()),
    int(test_predictions["p95_memory"].isna().sum()),
)

display(test_predictions.head())

Test occurrences predicted: 731
Test target columns available in this notebook: []
Missing P95 inputs (cpu/memory): 2 2


,occurrence_key,occurrence_id,recurrence_key,execution_start_time_us,prior_completed_occurrence_count,ml_pred_cpu,ml_pred_memory,p95_cpu,p95_memory
0,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,2303700000000,0,0.324071,0.068093,0.050171,0.010925
1,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,2664600000000,0,0.244887,0.105653,0.031437,0.030359
2,+OVSEukSmo6ma+28nY4raBvhviWrpQLsc+DWQcTGV4E=||...,+OVSEukSmo6ma+28nY4raBvhviWrpQLsc+DWQcTGV4E=||...,+OVSEukSmo6ma+28nY4raBvhviWrpQLsc+DWQcTGV4E=||...,2354603000000,0,2.918740,0.691192,40.361145,21.625664
3,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,2498100000000,10,0.266627,0.046251,1.320166,0.122806
4,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,+cz+wEXttqDjld+YWLmrgqExjdqiHMZ2WZODThyR9jE=||...,2594400000000,11,0.266701,0.040089,1.292969,0.114852


In [18]:
# Validation predictions from the selected models trained on primary training
# rows only, keyed to the common validation population.
val_predictions = val_eval[
    ["occurrence_key", "occurrence_id", "recurrence_key", "execution_start_time_us",
     "prior_completed_occurrence_count"]
].copy()

for resource, model_name in SELECTED_MODELS.items():
    val_predictions[f"actual_{resource}"] = val_eval[RESOURCE_TARGETS[resource]].to_numpy()
    val_predictions[f"ml_pred_{resource}"] = validation_predictions[
        (resource, "tuned", model_name)
    ]

val_predictions["p95_cpu"] = val_eval["rec_cpu_p95"].to_numpy()
val_predictions["p95_memory"] = val_eval["rec_memory_p95"].to_numpy()

assert set(val_predictions["occurrence_id"]) == VALIDATION_EVAL_IDS

display(val_predictions.head())

,occurrence_key,occurrence_id,recurrence_key,execution_start_time_us,prior_completed_occurrence_count,actual_cpu,ml_pred_cpu,actual_memory,ml_pred_memory,p95_cpu,p95_memory
0,HPPQSI0kZzeIvGJBaRD/DqDVKFEVdFseri0QSt2QEqM=||...,HPPQSI0kZzeIvGJBaRD/DqDVKFEVdFseri0QSt2QEqM=||...,HPPQSI0kZzeIvGJBaRD/DqDVKFEVdFseri0QSt2QEqM=||...,1813500000000,1,0.516968,1.328493,0.122406,0.442731,0.646404,0.587143
1,RtMF80uixRY5N+WdutV8L+5H1TSCXw1thAZdXIT+n+o=||...,RtMF80uixRY5N+WdutV8L+5H1TSCXw1thAZdXIT+n+o=||...,RtMF80uixRY5N+WdutV8L+5H1TSCXw1thAZdXIT+n+o=||...,1814100000000,0,0.556343,1.399616,7.211670,0.348413,NaN,NaN
2,ly4HE4d+6E8BqXgm6E1eGNKu8OiiZE+Gj/D4YUMRFDU=||...,ly4HE4d+6E8BqXgm6E1eGNKu8OiiZE+Gj/D4YUMRFDU=||...,ly4HE4d+6E8BqXgm6E1eGNKu8OiiZE+Gj/D4YUMRFDU=||...,1815814000000,1,0.000000,0.296753,0.000172,0.075820,0.060631,0.010214
3,dQgNDwOJCM2FNdmgBc3ZzN30l30yLFVmPOUDfKfmvDE=||...,dQgNDwOJCM2FNdmgBc3ZzN30l30yLFVmPOUDfKfmvDE=||...,dQgNDwOJCM2FNdmgBc3ZzN30l30yLFVmPOUDfKfmvDE=||...,1816500000000,2,0.024200,0.151682,0.002888,0.033001,0.120569,0.039602
4,mB48MswaSZ7cdHmzW4y0uUiCa2KTl0Fps1Mu8yeektw=||...,mB48MswaSZ7cdHmzW4y0uUiCa2KTl0Fps1Mu8yeektw=||...,mB48MswaSZ7cdHmzW4y0uUiCa2KTl0Fps1Mu8yeektw=||...,1819101000000,18,0.025543,1.242958,0.007027,0.235963,1.649455,0.226932


# Save Locked Models and ML Artifacts

In [19]:
def make_json_serializable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {
            key: make_json_serializable(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [make_json_serializable(item) for item in value]
    return value


feature_manifest = {
    "dataset": n04_metadata["dataset"],
    "canonical_modeling_unit": n04_metadata["canonical_modeling_unit"],
    "feature_order": FEATURE_COLS,
    "feature_dtype": "float64",
    "missing_value_handling": (
        "NaN preserved and passed to models with native missing-value support; "
        "no imputation and no zero-filling."
    ),
    "learned_preprocessing": "none",
    "leaky_columns": OCCURRENCE_LEAKY_COLUMNS,
    "leaky_prefixes": OCCURRENCE_LEAKY_PREFIXES,
    "target_cpu_field": TARGET_CPU_COL,
    "target_memory_field": TARGET_MEMORY_COL,
    "target_transform": "log1p for training; expm1 then clip at 0 for predictions",
    "label_period_rule": (
        "A label is used only when target_next_occurrence_start_time_us is at or "
        "before the end of the period the model is allowed to learn from "
        "(train_end for selection, val_end for the locked refit)."
    ),
    "source_config": "src/features/feature_config.py::OCCURRENCE_FEATURE_COLS",
    "source_helpers": "src/features/ml_prep.py",
}

LOCKED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
locked_model_paths = {}

for resource, model in locked_models.items():
    model_path = LOCKED_MODEL_DIR / f"{resource}_{SELECTED_MODELS[resource]}.joblib"
    joblib.dump(model, model_path)
    locked_model_paths[resource] = str(model_path.relative_to(REPO_ROOT))

n05_metadata = {
    "dataset": n04_metadata["dataset"],
    "notebook": "05_evaluateRecommendations.ipynb",
    "role": (
        "Tune and select final ML models, refit locked models, generate final ML "
        "predictions, and provide ML/P95 inputs for authoritative evaluation in N06."
    ),
    "test_metrics_computed": False,
    "test_target_columns_dropped_on_load": True,
    "hybrid_constructed": False,
    "input_artifacts": {
        "train": TRAIN_OCCURRENCES_PATH.name,
        "validation": VAL_OCCURRENCES_PATH.name,
        "test": TEST_OCCURRENCES_PATH.name,
        "n04_metadata": N04_METADATA_PATH.name,
        "n04_validation_comparison": N04_VALIDATION_COMPARISON_PATH.name,
    },
    "n04_selected_baseline_policy": SELECTED_BASELINE_POLICY,
    "baseline_min_runs": SELECTED_MIN_RUNS,
    "hybrid_safeguard_input": "p95_* = rec_*_p95 from N04 at baseline_min_runs",
    "feature_manifest": FEATURE_MANIFEST_PATH.name,
    "training_rows": {
        f"{resource}/{rule}": len(rows)
        for (resource, rule), rows in training_rows.items()
    },
    "common_validation_occurrences": len(val_eval),
    "random_state": RANDOM_STATE,
    "tuning": {
        "candidate_models": CANDIDATE_MODELS,
        "param_grids": PARAM_GRIDS,
        "fixed_params": FIXED_PARAMS,
        "cv": (
            f"purged_time_series_folds (TimeSeriesSplit, n_splits={N_CV_SPLITS}, "
            "timestamp-group preserving, training labels purged at each fold's "
            "validation start)"
        ),
        "scoring": "RMSE on log1p(target)",
        "robustness_check": f"GroupKFold(n_splits={N_CV_SPLITS}) by recurrence_key; not used for decisions",
        "best_params": {
            f"{resource}/{model_name}": params
            for (resource, model_name), params in best_params.items()
        },
    },
    "selection_rule": (
        "Per resource, lowest validation RMSE on the log1p scale among tuned "
        "candidates trained on train-period labels; tie-breaker lower raw MAE."
    ),
    "selected_models": SELECTED_MODELS,
    "locked_params": LOCKED_PARAMS,
    "refit": {
        "data": "train + validation occurrences",
        "label_cutoff": "target_next_occurrence_start_time_us <= val_end",
        "rows": {resource: len(rows) for resource, rows in refit_rows.items()},
    },
    "locked_models": locked_model_paths,
    "prediction_columns": ["ml_pred_cpu", "ml_pred_memory", "p95_cpu", "p95_memory"],
    "test_prediction_rows": len(test_predictions),
    "outputs": {
        "feature_manifest": FEATURE_MANIFEST_PATH.name,
        "tuning_results": TUNING_RESULTS_PATH.name,
        "cv_summary": CV_SUMMARY_PATH.name,
        "validation_model_comparison": MODEL_COMPARISON_PATH.name,
        "validation_predictions": VAL_PREDICTIONS_PATH.name,
        "test_predictions": TEST_PREDICTIONS_PATH.name,
    },
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    },
}

tuning_results.to_csv(TUNING_RESULTS_PATH, index=False)
cv_summary.to_csv(CV_SUMMARY_PATH, index=False)
model_comparison.to_csv(MODEL_COMPARISON_PATH, index=False)
val_predictions.to_parquet(VAL_PREDICTIONS_PATH, index=False)
test_predictions.to_parquet(TEST_PREDICTIONS_PATH, index=False)

with open(FEATURE_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(make_json_serializable(feature_manifest), file, indent=2)

with open(N05_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(make_json_serializable(n05_metadata), file, indent=2)

for path in [
    FEATURE_MANIFEST_PATH,
    TUNING_RESULTS_PATH,
    CV_SUMMARY_PATH,
    MODEL_COMPARISON_PATH,
    VAL_PREDICTIONS_PATH,
    TEST_PREDICTIONS_PATH,
    N05_METADATA_PATH,
]:
    print("Saved:", path)
for resource, path in locked_model_paths.items():
    print(f"Saved locked {resource} model:", path)

Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/ml_feature_manifest.json
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/ml_tuning_results.csv
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/ml_cv_summary.csv
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/ml_validation_model_comparison.csv
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/val_ml_predictions.parquet
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/test_ml_predictions.parquet
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/ml_metadata.json
Saved locked cpu model: data/processed/google_subset_v1/n05_locked_models/cpu_random_forest.joblib
Saved locked memory model: data/processed/google_subset_v1/n05_locked_models/memory_random_forest.joblib


## Saved Artifact Verification

In [20]:
saved_test = pd.read_parquet(TEST_PREDICTIONS_PATH)
saved_val = pd.read_parquet(VAL_PREDICTIONS_PATH)

pd.testing.assert_frame_equal(saved_test, test_predictions.reset_index(drop=True), check_dtype=False)
pd.testing.assert_frame_equal(saved_val, val_predictions.reset_index(drop=True), check_dtype=False)

assert saved_test["occurrence_id"].is_unique
assert not [column for column in saved_test.columns if column.startswith(("target_", "actual_"))]
assert set(saved_val["occurrence_id"]) == VALIDATION_EVAL_IDS

# Validation actuals stay aligned with the exact N04 targets.
target_check = saved_val.merge(
    val_occurrences[["occurrence_id", TARGET_CPU_COL, TARGET_MEMORY_COL]],
    on="occurrence_id",
    how="left",
    validate="one_to_one",
)
np.testing.assert_array_equal(target_check["actual_cpu"], target_check[TARGET_CPU_COL])
np.testing.assert_array_equal(target_check["actual_memory"], target_check[TARGET_MEMORY_COL])

assert len(pd.read_csv(TUNING_RESULTS_PATH)) == len(tuning_results)
assert len(pd.read_csv(CV_SUMMARY_PATH)) == len(cv_summary)
assert len(pd.read_csv(MODEL_COMPARISON_PATH)) == len(model_comparison)

with open(FEATURE_MANIFEST_PATH, "r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
with open(N05_METADATA_PATH, "r", encoding="utf-8") as file:
    saved_metadata = json.load(file)

assert saved_manifest["feature_order"] == FEATURE_COLS
assert saved_metadata["selected_models"] == SELECTED_MODELS
assert saved_metadata["test_metrics_computed"] is False

# Reloaded locked models reproduce the saved test predictions.
for resource, model_path in locked_model_paths.items():
    reloaded = joblib.load(REPO_ROOT / model_path)
    np.testing.assert_allclose(
        predict_resource(reloaded, test_features),
        saved_test[f"ml_pred_{resource}"],
        rtol=1e-6,
        atol=1e-12,
    )

print("Saved artifact, metadata, and locked-model reload verification passed.")

Saved artifact, metadata, and locked-model reload verification passed.


# N05 Handoff to N06

After this notebook runs successfully:

- N05 consumed the N04 occurrence artifacts without rebuilding splits or targets;
- the feature manifest (`ml_feature_manifest.json`, mirrored in `src/features/feature_config.py`) contains only start-of-occurrence information and passed the leakage audit on train, validation, and test;
- missing values were preserved, with no imputation or zero-filling;
- CPU and memory models were trained independently on labels inside the allowed period; the all-labels variant is reported only as a sensitivity check;
- all N04 baseline validation metrics were reproduced exactly;
- Decision Tree, Random Forest, and XGBoost were tuned with label-purged `TimeSeriesSplit`; `GroupKFold` was reported separately and not used for decisions;
- one final model per resource was selected on validation with a pre-declared rule, then refit on train + validation;
- `test_ml_predictions.parquet` contains `ml_pred_cpu`, `ml_pred_memory`, `p95_cpu`, and `p95_memory` for every test occurrence, keyed by `occurrence_id`, with no target or actual columns;
- `val_ml_predictions.parquet` contains the same columns plus `actual_*` for the common validation population, from models trained on training rows only;
- no test metric was computed and no Hybrid recommendation was constructed.

**N06** should load `test_ml_predictions.parquet` and `ml_metadata.json`, join the N04 test targets by `occurrence_id`, construct Hybrid from `ml_pred_*` and `p95_*`, and compare Request, Mean, P90, P95, P99, ML, and Hybrid through the shared evaluator.